# Mental Health Chatbot — Sentiment Model Training
Run each cell in order. This trains a TF-IDF + Logistic Regression classifier on the GoEmotions dataset.

In [ ]:
# Step 1: Install dependencies (run once)
!pip install pandas scikit-learn nltk joblib

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
import os
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

nltk.download('stopwords')
print('Libraries loaded successfully!')

In [ ]:
# Step 2: Load GoEmotions dataset
# Download from: https://github.com/google-research/google-research/tree/master/goemotions/data
# Place train.tsv, dev.tsv, test.tsv in the same folder as this notebook

# GoEmotions columns: text, emotion_ids, id
# Emotion IDs reference: https://github.com/google-research/google-research/blob/master/goemotions/data/emotions.txt

# For simplicity, we map GoEmotions 27 labels -> 5 core labels
EMOTION_MAP = {
    0: 'happy',    # admiration
    1: 'happy',    # amusement
    2: 'anxious',  # anger -> anxious (overlaps in stress context)
    3: 'sad',      # annoyance
    4: 'happy',    # approval
    5: 'happy',    # caring
    6: 'anxious',  # confusion
    7: 'happy',    # curiosity
    8: 'happy',    # desire
    9: 'sad',      # disappointment
    10: 'sad',     # disapproval
    11: 'sad',     # disgust
    12: 'anxious', # embarrassment
    13: 'happy',   # excitement
    14: 'anxious', # fear
    15: 'happy',   # gratitude
    16: 'sad',     # grief
    17: 'happy',   # joy
    18: 'happy',   # love
    19: 'sad',     # nervousness -> sad
    20: 'neutral', # neutral
    21: 'happy',   # optimism
    22: 'sad',     # pride -> mapped loosely
    23: 'happy',   # realization
    24: 'sad',     # relief -> sad
    25: 'sad',     # remorse
    26: 'sad',     # sadness
    27: 'anxious', # surprise -> anxious
}

def load_goemotions(filepath):
    df = pd.read_csv(filepath, sep='\t', header=None, names=['text', 'emotion_ids', 'id'])
    # Take the first emotion id if multiple
    df['emotion_id'] = df['emotion_ids'].apply(lambda x: int(str(x).split(',')[0]))
    df['emotion'] = df['emotion_id'].map(EMOTION_MAP).fillna('neutral')
    return df[['text', 'emotion']]

# Load all splits
try:
    train_df = load_goemotions('train.tsv')
    dev_df   = load_goemotions('dev.tsv')
    test_df  = load_goemotions('test.tsv')
    df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
    print(f'Dataset loaded: {len(df)} samples')
    print(df['emotion'].value_counts())
except FileNotFoundError:
    # Fallback: generate a small demo dataset so the notebook runs end-to-end
    print('GoEmotions files not found. Using built-in demo dataset.')
    demo_data = {
        'text': [
            'I feel so happy today!', 'This is amazing, I love it!', 'Great news, I am thrilled!',
            'I am really sad and lonely', 'Nothing is going right for me', 'I feel hopeless',
            'I am so anxious about my exams', 'I cannot stop worrying', 'Everything feels overwhelming',
            'I am furious at this situation', 'This makes me so angry', 'I hate how things turned out',
            'Just another regular day', 'Nothing special happened', 'I went to college today',
        ],
        'emotion': [
            'happy','happy','happy',
            'sad','sad','sad',
            'anxious','anxious','anxious',
            'angry','angry','angry',
            'neutral','neutral','neutral',
        ]
    }
    df = pd.DataFrame(demo_data)
    print(f'Demo dataset: {len(df)} samples')
    print(df['emotion'].value_counts())

In [ ]:
# Step 3: Text Preprocessing
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)              # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len() > 2]  # drop empty rows

print('Sample cleaned texts:')
print(df[['text','clean_text','emotion']].head(5).to_string())

In [ ]:
# Step 4: Train/Test Split
X = df['clean_text']
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# Step 5: Build and Train Pipeline (TF-IDF + Logistic Regression)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),   # unigrams and bigrams
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=5.0,
        solver='lbfgs',
        multi_class='multinomial'
    ))
])

pipeline.fit(X_train, y_train)
print('Model trained!')

In [ ]:
# Step 6: Evaluate
y_pred = pipeline.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}\n')
print(classification_report(y_test, y_pred))

In [ ]:
# Step 7: Save the model
os.makedirs('../backend/model', exist_ok=True)
joblib.dump(pipeline, '../backend/model/sentiment_model.pkl')
print('Model saved to ../backend/model/sentiment_model.pkl')

# Quick test
test_sentences = [
    'I feel so stressed about my exams',
    'Today was a wonderful day!',
    'I am really sad and alone',
    'Nothing is going well for me',
]
cleaned = [clean_text(s) for s in test_sentences]
predictions = pipeline.predict(cleaned)
print('\nSample predictions:')
for sent, pred in zip(test_sentences, predictions):
    print(f'  "{sent}" -> {pred}')